In [ ]:
# Standard library imports
import os
import sys

# Add src to path
sys.path.append(os.path.join(os.getcwd(), 'src'))

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local imports
from src.data_loader import DataLoader
from src.data_processor import DataProcessor
from src.model_trainer import ModelTrainer, calculate_top_k_accuracy
from src.predictor import MusicPredictor
from src.visualizer import Visualizer

print("All modules imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
"""Configuration parameters for the pipeline."""

# Directories
RAW_DATA_DIR = "data/raw/"
PROCESSED_DATA_DIR = "data/processed/"
MODEL_DIR = "models/"
ARTIFACTS_DIR = "artifacts/"

# Data processing parameters
GENRE_THRESHOLD = 1000  # Minimum tag occurrences to be considered a genre
TOP_N_ARTISTS = 20      # Number of top artists per user to consider

# Model parameters
HIDDEN_LAYERS = [256, 128, 64]
DROPOUT_RATES = [0.3, 0.2, 0.1]
EPOCHS = 50
BATCH_SIZE = 64
RANDOM_STATE = 42

# Paths
MODEL_PATH = os.path.join(MODEL_DIR, "multilabel_model.h5")

print("Configuration set!")
print(f"Model will be saved to: {MODEL_PATH}")

In [ ]:
"""
Download and load the Last.fm dataset
"""

# Initialize data loader
loader = DataLoader(raw_dir=RAW_DATA_DIR)

# Download and extract dataset
try:
    loader.download_dataset()
    loader.extract_dataset()
    print("\nDataset downloaded and extracted successfully!")
except Exception as e:
    print(f"Error during data loading: {e}")
    print("Please check your internet connection and try again.")
    raise

# Load raw data files
try:
    user_artists, demographics, artists, user_tags, tags = loader.load_raw_data()
    print("\nData loaded successfully!")
    print(f"User artists shape: {user_artists.shape}")
    print(f"Demographics shape: {demographics.shape}")
    print(f"Artists shape: {artists.shape}")
    print(f"User tags shape: {user_tags.shape}")
    print(f"Tags shape: {tags.shape}")
except Exception as e:
    print(f"Error loading data files: {e}")
    raise

# Quick data exploration
print("\n" + "="*60)
print("Quick Data Preview")
print("="*60)
print("\nUser Artists Sample:")
print(user_artists.head())
print("\nDemographics Sample:")
print(demographics.head())

In [ ]:
"""
Process data and extract genre information
"""

# Initialize processor (inherits from DataLoader)
processor = DataProcessor(
    raw_dir=RAW_DATA_DIR,
    processed_dir=PROCESSED_DATA_DIR,
    genre_threshold=GENRE_THRESHOLD,
    top_n_artists=TOP_N_ARTISTS
)

# Extract genres from tags
print("\n1. Extracting genre information...")
artist_tags_df = processor.extract_genres(tags, user_tags)
print(f"Artist tags extracted: {artist_tags_df.shape}")

# Filter to common genres
print("\n2. Filtering common genres...")
common_genres = processor.filter_common_genres(artist_tags_df)
print(f"Found {len(common_genres)} common genres")
print(f"Sample genres: {common_genres[:10]}")

# Process user-genre relationships
print("\n3. Processing user-genre relationships...")
user_genre_multi = processor.process_user_genres(
    user_artists,
    artist_tags_df,
    common_genres
)
print(f"User-genre relationships: {user_genre_multi.shape}")

# Prepare features and labels
print("\n4. Preparing features and labels...")
X, Y, genre_classes = processor.prepare_features_and_labels(
    user_genre_multi,
    demographics
)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Label matrix shape: {Y.shape}")
print(f"Number of genres: {len(genre_classes)}")
print(f"Genre classes: {genre_classes}")

# Save processed data
print("\n5. Saving processed data...")
processor.save_processed_data(X, Y, genre_classes)
print("Processed data saved successfully!")

In [ ]:
"""
Build and train the neural network model
"""

# Initialize model trainer
trainer = ModelTrainer(input_dim=X.shape[1], output_dim=Y.shape[1])

# Build model
print("\n1. Building model...")
trainer.build_model(hidden_layers=HIDDEN_LAYERS, dropout_rates=DROPOUT_RATES)
trainer.summary()

# Train model
print("\n2. Training model...")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")

data_splits = trainer.train(
    X, Y,
    test_size=0.3,
    val_size=0.5,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    model_save_path=MODEL_PATH,
    random_state=RANDOM_STATE
)

print("\nModel training complete!")

# Extract splits for evaluation
X_train = data_splits['X_train']
y_train = data_splits['y_train']
X_val = data_splits['X_val']
y_val = data_splits['y_val']
X_test = data_splits['X_test']
y_test = data_splits['y_test']

In [ ]:
"""
Evaluate model performance
"""

# Evaluate on test set
print("\n1. Evaluating on test set...")
metrics = trainer.evaluate(X_test, y_test)

# Calculate top-k accuracies
print("\n2. Calculating top-k accuracies...")
y_prob_test = trainer.model.predict(X_test)

for k in [1, 3, 5]:
    acc = calculate_top_k_accuracy(y_test, y_prob_test, k=k)
    print(f"Top-{k} accuracy: {acc:.4f}")

# Visualize training history
print("\n3. Visualizing training history...")
visualizer = Visualizer(predictor=None)  # Will set predictor later
visualizer.plot_training_history(trainer.history, figsize=(12, 5))

In [ ]:
"""
Make predictions for new users
"""

# Load predictor
print("\n1. Loading predictor...")
predictor = MusicPredictor.load_model(MODEL_PATH, ARTIFACTS_DIR)

# Update visualizer with predictor
visualizer = Visualizer(predictor)

# Example users for prediction
example_users = [
    {
        'name': 'Young Male US User',
        'data': {'age': 23, 'gender': 'm', 'country': 'United States', 'registered': 2010}
    },
    {
        'name': 'Older Female UK User',
        'data': {'age': 45, 'gender': 'f', 'country': 'United Kingdom', 'registered': 2008}
    },
    {
        'name': 'Middle-aged User from Germany',
        'data': {'age': 35, 'gender': 'm', 'country': 'Germany', 'registered': 2011}
    }
]

# Make predictions for each user
print("\n2. Predicting genres for example users...")
print("="*60)

for user_info in example_users:
    print(f"\n{user_info['name']}:")
    print(f"Demographics: {user_info['data']}")
    print("\nTop-5 Predicted Genres:")

    predictions = predictor.predict(user_info['data'], top_k=5)

    # Use enumerate to display results
    for i, (genre, prob) in enumerate(predictions, 1):
        print(f"  {i}. {genre}: {prob:.3f}")

    print()

In [ ]:
"""
Visualize predictions
"""

# Visualize predictions for first example user
print("\nVisualizing predictions for:", example_users[0]['name'])
visualizer.plot_top_k_predictions(example_users[0]['data'], k=5)

# Show all genres for the user
print("\nShowing all genre probabilities...")
visualizer.plot_all_genres(example_users[0]['data'], figsize=(12, 10))

In [ ]:
"""
Interactive prediction function for custom users
"""

def predict_for_custom_user(age: int, gender: str, country: str,
                           registered: int, top_k: int = 5):
    """
    Predict genres for a custom user with given demographics.

    Args:
        age: User's age
        gender: User's gender ('m' or 'f')
        country: User's country
        registered: Year user registered
        top_k: Number of top genres to return

    Returns:
        List of (genre, probability) tuples
    """
    user_dict = {
        'age': age,
        'gender': gender,
        'country': country,
        'registered': registered
    }

    predictions = predictor.predict(user_dict, top_k=top_k)

    print(f"\nUser Profile:")
    print(f"  Age: {age}")
    print(f"  Gender: {gender}")
    print(f"  Country: {country}")
    print(f"  Registered: {registered}")
    print(f"\nTop-{top_k} Predicted Genres:")

    # Use for loop with enumerate
    for i, (genre, prob) in enumerate(predictions, 1):
        print(f"  {i}. {genre}: {prob:.3f} ({prob*100:.1f}%)")

    # Visualize
    visualizer.plot_top_k_predictions(user_dict, k=top_k)

    return predictions

# Example usage
print("Example: Predicting for a 28-year-old male from Canada")
custom_predictions = predict_for_custom_user(
    age=28,
    gender='m',
    country='Canada',
    registered=2012,
    top_k=5
)